# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains structured experimental data from human mass spectrometry analyses.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset and show the main metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (using Croissant `@id` for unambiguous reference).

In [ ]:
# List all available record sets and their @id
print('Available Record Sets:')
for record_set in metadata.record_sets:
    print(f"- Name: {getattr(record_set, 'name', '(unnamed)')} | @id: {record_set.id}")
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - {getattr(field, 'name', '(unnamed)')} | @id: {field.id} | data_type: {getattr(field, 'data_type', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Use the record set and field `@id`s from the overview above.

In [ ]:
# List all record set @ids
record_sets = [record_set.id for record_set in metadata.record_sets]
print('All RecordSet @ids:', record_sets)

# Load each record set into a Pandas DataFrame, using `@id` as keys
dataframes = {}
for rs_id in record_sets:
    print(f"Loading records from RecordSet @id: {rs_id}")
    try:
        # List of dicts; each dict represents a record
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print(f"  No records returned for @id: {rs_id}")
    except Exception as e:
        print(f"  Failed to load @id: {rs_id}: {e}")

# Choose the primary record set for analysis (using the first one here as example)
main_record_set_id = record_sets[0]
print(f"\nMain analysis RecordSet @id: {main_record_set_id}")
df_main = dataframes[main_record_set_id]
print('Fields/Columns:', df_main.columns.tolist())
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps such as filtering, normalization, and grouping.

**Note:** All fields are referenced by their `@id`, as per FAIR best practices.

In [ ]:
# Example: pick a numeric field (e.g., protein abundance, MW) by @id
# (Adjust the @id variable below to match your dataset if necessary.)

# Inspect available columns again
print('Available numeric fields:')
for col in df_main.columns:
    if pd.api.types.is_numeric_dtype(df_main[col]):
        print(col)

# Select a numeric field id from those listed above
numeric_field_id = df_main.select_dtypes('number').columns[0]  # as example, pick the first numeric
print(f"Using numeric field @id: {numeric_field_id}")

threshold = df_main[numeric_field_id].quantile(0.75)  # e.g., keep top 25%
filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (top 25%):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if available
potential_group_fields = df_main.select_dtypes(include=['object', 'category']).columns.tolist()
print('Potential group-by fields:', potential_group_fields)

if potential_group_fields:
    group_field_id = potential_group_fields[0]
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id]
        .mean()
        .reset_index()
        .sort_values(numeric_field_id, ascending=False)
    )
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print('No suitable group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields. All fields are referenced by @id; replace as-needed for your use case.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df_main[numeric_field_id], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping worked, show a barplot
if potential_group_fields:
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
This notebook demonstrated exploring the FAIR² protein mass spectrometry dataset using Croissant. You loaded record sets, referenced every entity by `@id`, and performed basic analysis and visualization.

- To go further, see Croissant documentation and schema definitions for advanced linking, more EDA, or ML tasks.
- All field access and transformations follow Croissant's best practices for schema referencing and reproducibility.